### Synthetic Data Generator using Llama (via Ollama)

This notebook uses a local Llama model (through Ollama) to generate synthetic datasets in CSV format, based on a description of the dataset and its fields.

In [1]:
import csv
import io
import pandas as pd
from IPython.display import Markdown, display
import ollama

In [2]:
MODEL = "llama3.2"

### Step 1: Describe the dataset you want to generate

Set the business context, the columns you want, and how many rows to generate.

In [3]:
dataset_description = "A dataset of customer orders for an online electronics store"

columns = {
    "order_id": "a unique order identifier, e.g. ORD-10001",
    "customer_name": "a realistic full name",
    "product": "the name of an electronics product",
    "quantity": "an integer between 1 and 5",
    "price": "the unit price in USD, a decimal number",
    "order_date": "a date in the last 12 months, format YYYY-MM-DD"
}

num_rows = 20

### Step 2: Build the prompts

In [4]:
system_prompt = "You are a data generation assistant that creates realistic, diverse synthetic datasets. \
You always respond with raw CSV data only - no markdown formatting, no code fences, and no extra commentary. \
The first row must be the header row with the exact column names provided."

In [5]:
def user_prompt_for(description, columns, num_rows):
    column_lines = "\n".join(f"- {name}: {hint}" for name, hint in columns.items())
    prompt = f"Generate {num_rows} rows of synthetic data for the following dataset:\n"
    prompt += f"{description}\n\n"
    prompt += "The columns are:\n"
    prompt += column_lines
    prompt += "\n\nReturn the data as CSV only, starting with the header row of column names, "
    prompt += "with one row per record and no extra text before or after the CSV."
    return prompt

In [6]:
def messages_for(description, columns, num_rows):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(description, columns, num_rows)}
    ]

### Step 3: Generate the synthetic data

In [7]:
def generate_synthetic_csv(description, columns, num_rows):
    response = ollama.chat(model=MODEL, messages=messages_for(description, columns, num_rows))
    content = response['message']['content']

    # Strip code fences if the model added them anyway
    content = content.strip()
    if content.startswith("```"):
        content = content.split("```")[1]
        if content.startswith("csv"):
            content = content[len("csv"):]
        content = content.strip()

    return content

In [8]:
csv_text = generate_synthetic_csv(dataset_description, columns, num_rows)
print(csv_text)

order_id,customer_name,product,quantity,price,order_date
ORD-10001,Alexander Brown,Laptop,2,999.99,2024-03-15
ORD-10002,Evelyn Davis,Smartphone,5,599.95,2024-04-10
ORD-10003,Benjamin White,Tablet,1,399.00,2023-11-22
ORD-10004,Maria Rodriguez,Laptop,4,899.90,2024-02-12
ORD-10005,William Harris,Headphones,3,79.99,2024-01-14
ORD-10006,Samantha Lee,Smartwatch,1,299.50,2023-10-28
ORD-10007,Ava Martin,Laptop,2,899.90,2024-03-25
ORD-10008,Noah Taylor,Tablet,5,499.95,2023-09-22
ORD-10009,Isabella Walker,Smartphone,1,599.99,2024-04-05
ORD-10010,Liam Hall,Headphones,4,149.00,2023-07-30
ORD-10011,Elijah Brown,Smartwatch,3,299.95,2024-01-17
ORD-10012,Hannah White,Tablet,2,799.50,2024-03-20
ORD-10013,James Davis,Laptop,5,1199.99,2023-05-27
ORD-10014,Ava Rodriguez,Smartphone,1,599.95,2024-04-21
ORD-10015,Noah Martin,Headphones,2,79.99,2024-02-06
ORD-10016,Sofia Lee,Laptop,3,399.00,2023-08-27
ORD-10017,Benjamin Walker,Tablet,4,899.90,2023-11-15
ORD-10018,Ava Hall,Smartwatch,2,299.50,2024-03-14
ORD-10

### Step 4: Load into a DataFrame and save to file

In [9]:
df = pd.read_csv(io.StringIO(csv_text))
df

,order_id,customer_name,product,quantity,price,order_date
0,ORD-10001,Alexander Brown,Laptop,2,999.99,2024-03-15
1,ORD-10002,Evelyn Davis,Smartphone,5,599.95,2024-04-10
2,ORD-10003,Benjamin White,Tablet,1,399.00,2023-11-22
3,ORD-10004,Maria Rodriguez,Laptop,4,899.90,2024-02-12
4,ORD-10005,William Harris,Headphones,3,79.99,2024-01-14
5,ORD-10006,Samantha Lee,Smartwatch,1,299.50,2023-10-28
6,ORD-10007,Ava Martin,Laptop,2,899.90,2024-03-25
7,ORD-10008,Noah Taylor,Tablet,5,499.95,2023-09-22
8,ORD-10009,Isabella Walker,Smartphone,1,599.99,2024-04-05
9,ORD-10010,Liam Hall,Headphones,4,149.00,2023-07-30


In [ ]:
df.to_csv("synthetic_data.csv", index=False)
print("Saved to synthetic_data.csv")

Saved to synthetic_data.csv
